In [ ]:
from bak.datasets import StrawberryDataset
from matplotlib import pyplot as plt

data_ds = StrawberryDataset(r"strawberry_cls\images")
image, labels = data_ds[0]

print(image.max(), image.min())

plt.imshow(image.permute(1, 2, 0))
plt.axis('off')
plt.show()

# for label in labels:
#     class_id, x_min, y_min, x_max, y_max = label
#     x_min, y_min, x_max, y_max = list(map(lambda x: x * 640, [x_min, y_min, x_max, y_max]))
#     plt.gca().add_patch(plt.Rectangle((x_min, y_min), x_max - x_min, y_max - y_min,
#                                       edgecolor='red', facecolor='none', linewidth=2))
#     plt.text(x_min, y_min - 10, str(class_id), color='red', fontsize=12)
# plt.show()

In [ ]:
# def polygan2box(text, format='xyxy'):
#     text = text.replace("\n", "").split(" ")
#     class_id = int(text[0])
#     poly = list(map(float, text[1:]))
    
#     xs = poly[::2]
#     ys = poly[1::2]
#     x_min, x_max = min(xs), max(xs)
#     y_min, y_max = min(ys), max(ys)
#     if format == 'xyxy':
#         return [class_id, x_min, y_min, x_max, y_max]
#     elif format == 'xywh':
#         return [class_id, x_min, y_min, x_max - x_min, y_max - y_min]
#     elif format == 'cxcywh':
#         return [class_id, (x_min + x_max) / 2, (y_min + y_max) / 2, x_max - x_min, y_max - y_min]

# import glob 
# import os

# labels = glob.glob("strawberry_cls/labels.init/*.txt")
# new_folder = "strawberry_cls/labels"

# os.makedirs(new_folder, exist_ok=True)

# for label in labels:
#     with open(label, "r") as f:
#         lines = f.readlines()
        
#     new_label_path = os.path.join(new_folder, os.path.basename(label))
#     with open(new_label_path, "w") as f:
#         for line in lines:
#             box = polygan2box(line, format='cxcywh')
#             f.write(" ".join(map(str, box)) + "\n")
            

In [ ]:
# import os
# import glob
# import random
# import shutil
# from os import path as osp

# ds_path = "strawberry_cls/"

# train_path = osp.join(ds_path, "train")
# val_path = osp.join(ds_path, "val")
# test_path = osp.join(ds_path, "test")

# for path in [train_path, val_path, test_path]:
#     os.makedirs(path, exist_ok=True)
#     os.makedirs(osp.join(path, "images"), exist_ok=True)
#     os.makedirs(osp.join(path, "labels"), exist_ok=True)

# images = glob.glob(osp.join(ds_path, "images/*.jpg"))
# random.shuffle(images)
# num_images = len(images)
# num_train = int(0.7 * num_images)
# num_val = int(0.15 * num_images)

# train_images = images[:num_train]
# val_images = images[num_train:num_train + num_val]
# test_images = images[num_train + num_val:]

# for img_path in train_images:
#     shutil.copy(img_path, osp.join(train_path, "images"))
#     shutil.copy(img_path.replace("images", "labels").replace(".jpg", ".txt"), osp.join(train_path, "labels"))
    
# for img_path in val_images:
#     shutil.copy(img_path, osp.join(val_path, "images"))
#     shutil.copy(img_path.replace("images", "labels").replace(".jpg", ".txt"), osp.join(val_path, "labels"))

# for img_path in test_images:
#     shutil.copy(img_path, osp.join(test_path, "images"))
#     shutil.copy(img_path.replace("images", "labels").replace(".jpg", ".txt"), osp.join(test_path, "labels"))


In [ ]:
import yaml
import torch

from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils import ops

d = yaml.load(open("yolov12.yaml", "r"), Loader=yaml.FullLoader)
d["scale"] = "n"
model = DetectionModel(cfg=d, ch=3, verbose=False)

model_in = torch.randn(1, 3, 640, 640)

# train model only outputs predictions
model.train()
out = model(model_in)
print(type(out))
for item in out:
    print(type(item), item.shape)

# eval model outputs predictions and features
model.eval()
pred, feature = model(model_in)
print(type(pred), type(feature))
print(pred.shape)
for item in feature:
    print(type(item), item.shape)

# NMS
out = ops.non_max_suppression(pred, conf_thres=0.25, iou_thres=0.45)
print(type(out), len(out))
for item in out:
    print(type(item), item.shape)

#### **测试代码**

In [ ]:
import glob
import yaml
import torch
import numpy as np

from tqdm import tqdm
from torch.utils.data import DataLoader

from bak.datasets import StrawberryDataset
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils import ops
from types import SimpleNamespace
from metrics import mean_ap

device = torch.device("cpu")

default_cfg = yaml.load(open("ultralytics/cfg/default.yaml", 'r'), Loader=yaml.FullLoader)
model_cfg = yaml.load(open("cfgs/yolov12.yaml", "r"), Loader=yaml.FullLoader)
model_cfg["scale"] = "n"

model = DetectionModel(cfg=model_cfg, ch=3, verbose=False).to(device)
model.args = SimpleNamespace(**{**default_cfg, **model_cfg})
# model.load_state_dict(torch.load("ckpts/yolov12.pt", map_location=device)["model"])
model.eval()

images = glob.glob("strawberry_cls/images/*.jpg")
np.random.shuffle(images)
test_images = images[int(0.8 * len(images)):]
print(f"Test images: {len(test_images)}")
data_ds = StrawberryDataset(test_images)
data_dl = DataLoader(data_ds, batch_size=1, shuffle=False, collate_fn=StrawberryDataset.collate_fn)


pred_list = []
target_list = []

for batch in tqdm(data_dl, desc="Testing", dynamic_ncols=True):
    model_in = batch["img"].to(device)
    with torch.no_grad():
        pred, _ = model(model_in)
    pred = ops.non_max_suppression(pred, conf_thres=0.25, iou_thres=0.45)
    
    batch_size = model_in.shape[0]
    img_h, img_w = model_in.shape[2], model_in.shape[3]

    for i in range(batch_size):
        pred_i = pred[i].detach().cpu().numpy() if len(pred) > i else np.zeros((0, 6), dtype=np.float32)
        pred_list.append(pred_i)

        idx = batch["batch_idx"] == i
        cls = batch["cls"][idx].detach().cpu().numpy().reshape(-1)
        bboxes = batch["bboxes"][idx].detach().cpu().numpy()

        if len(bboxes):
            bboxes_xyxy = ops.xywh2xyxy(torch.from_numpy(bboxes)).numpy()
            bboxes_xyxy[:, [0, 2]] *= img_w
            bboxes_xyxy[:, [1, 3]] *= img_h
            targets_i = np.column_stack([cls, bboxes_xyxy]).astype(np.float32)
        else:
            targets_i = np.zeros((0, 5), dtype=np.float32)

        target_list.append(targets_i)

    num_classes = 3
    metrics = mean_ap(pred_list, target_list, num_classes=num_classes)

print(f"Precision: {metrics['precision']:.4f}")
print(f"Recall: {metrics['recall']:.4f}")
print(f"F1-score: {metrics['f1']:.4f}")
print(f"mAP@0.5: {metrics['map50']:.4f}")
print(f"mAP@0.5:0.95: {metrics['map50_95']:.4f}")

In [ ]:
import torch
import sys

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Torch CUDA:", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("cuDNN:", torch.backends.cudnn.version())

### 导出

In [ ]:
import yaml
import torch

from ultralytics.nn.tasks import DetectionModel


default_cfg = yaml.load(open("ultralytics/cfg/default.yaml", "r"), Loader=yaml.FullLoader)
model_cfg = yaml.load(open("cfgs/yolov12.yaml", "r"), Loader=yaml.FullLoader)
model_cfg["scale"] = "n"

model = DetectionModel(cfg=model_cfg, ch=3, verbose=False)
pth = torch.load("ckpts/model_epoch_600.pth", map_location="cpu")
model.load_state_dict(pth["model"])
# model.fuse(True)
model = model.eval()

dummy_input = torch.randn(1, 3, 640, 640)
torch.onnx.export(
    model,
    dummy_input,
    "onnx/test/yolov12.onnx",
    opset_version=18,
    input_names=["images"],
    output_names=["output0"],
    dynamic_axes=None,
    external_data=False
)

# simplify ONNX model
import onnx
import onnxslim
from onnxslim.argparser import OnnxSlimKwargs

model_onnx = onnx.load("onnx/test/yolov12.onnx")
slim_args = OnnxSlimKwargs(
    input_shapes={"images": [1, 3, 640, 640]},
    no_shape_infer=True
)
model_onnx = onnxslim.slim(model_onnx)
onnx.save(model_onnx, "onnx/test/yolov12_simplified.onnx")

Model summary (fused): 497 layers, 2,508,929 parameters, 2,508,913 gradients, 5.8 GFLOPs
[torch.onnx] Obtain model graph for `DetectionModel([...]` with `torch.export.export(..., strict=False)`...


/root/autodl-tmp/envs/yolo/lib/python3.10/contextlib.py:142: UserWarning: The tensor attributes self.model.21.strides, self.model.21.anchors were assigned during export. Such attributes must be registered as buffers using the `register_buffer` API (https://pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.register_buffer).
  next(self.gen)


[torch.onnx] Obtain model graph for `DetectionModel([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ❌


ConversionError: Failed to decompose the FX graph for ONNX compatibility. [96mThis is step 2/3[0m of exporting the model to ONNX. Next steps:
- Create an issue in the PyTorch GitHub repository against the [96m*torch.export*[0m component and attach the full error stack as well as reproduction scripts.
- Create an error report with `torch.onnx.export(..., report=True)`, and save the ExportedProgram as a pt2 file. Create an issue in the PyTorch GitHub repository against the [96m*onnx*[0m component. Attach the error report and the pt2 model.

## Exception summary

<class 'ValueError'>: Cannot assign non-leaf Tensor to parameter 'weight'. Model parameters must be created explicitly. To express 'weight' as a function of another Tensor, compute the value in the forward() method.

(Refer to the full stack trace above for more information.)

### 测试 onnx

In [ ]:
import onnx
import random
import numpy as np
import onnxruntime as ort
import glob

from utils import sigmoid, xywh2xyxy, nms
from PIL import Image
from matplotlib import pyplot as plt

session = ort.InferenceSession("onnx/model_epoch_600_saved_model/model_epoch_600.onnx")
inputs = session.get_inputs()
print("Input name:", inputs[0].name)
print("Input shape:", inputs[0].shape)
print("Input type:", inputs[0].type)

# path = "strawberry_cls/test/images/20250831_101627_jpg.rf.baa9958c0e183de4acc059f56bd3e0d3.jpg"
imgs = glob.glob("strawberry_cls/test/images/*.jpg")
path = imgs[random.randint(0, len(imgs) - 1)]
image = np.asarray(Image.open(path).convert("RGB")).copy()
image = image.astype(np.float32).transpose(2, 0, 1) / 255.0
image = np.expand_dims(image, axis=0)

outputs = session.run(None, {inputs[0].name: image})
anchors = outputs[0][0].transpose(1, 0)

# split
boxes, cls = anchors[:, :4], anchors[:, 4:]
boxes = xywh2xyxy(boxes)
conf = cls.max(axis=1)
cls_id = cls.argmax(axis=1)

# filter
mask = conf > 0.25
boxes = boxes[mask]
cls = cls[mask]
conf = conf[mask]
cls_id = cls_id[mask]

# nms
pred_idx = nms(boxes, conf, iou_threshold=0.45)

# print(pred_idx)
# print(boxes[pred_idx])
# print(cls[pred_idx])
# print(conf[pred_idx])

cls2label = {0: "fullripe", 1: "semiripe", 2: "unripe"}
plt.imshow(image[0].transpose(1, 2, 0))
for i in pred_idx:
    x_min, y_min, x_max, y_max = boxes[i]
    plt.gca().add_patch(plt.Rectangle((x_min, y_min), x_max - x_min, y_max - y_min,
                                      edgecolor='red', facecolor='none', linewidth=2))
    plt.text(x_min, y_min - 10, f"{cls2label[cls_id[i]]} {conf[i]:.2f}", color='red', fontsize=12)

In [ ]:
# python: 检查 onnx 输入和值信息并找出含 3200 的维度
import onnx
m = onnx.load("onnx/yolov12_simplified.onnx")
onnx.checker.check_model(m)
def dims_of(vi):
    dims=[]
    t=vi.type.tensor_type
    for d in t.shape.dim:
        if d.HasField("dim_value"):
            dims.append(d.dim_value)
        elif d.HasField("dim_param"):
            dims.append(d.dim_param)
        else:
            dims.append(None)
    return dims

for vi in list(m.graph.input)+list(m.graph.value_info)+list(m.graph.output):
    try:
        ds = dims_of(vi)
    except Exception:
        continue
    if 3200 in ds:
        print("Found 3200 in:", vi.name, ds)

print("Model inputs:")
for i in m.graph.input:
    print(i.name, dims_of(i))

### 量化数据

In [ ]:
import os
import glob
import numpy as np
from PIL import Image

image_paths = sorted(glob.glob("strawberry_cls/train/images/*.jpg"))
if not image_paths:
    raise RuntimeError("No images found in strawberry_cls/train/images")

images = []
for image_path in image_paths:
    image = Image.open(image_path).convert("RGB").resize((640, 640))
    image_array = np.asarray(image, dtype=np.float32)  # 0~255
    images.append(image_array)

train_images = np.stack(images, axis=0).astype(np.float32)  # NHWC
os.makedirs("onnx", exist_ok=True)
np.save("onnx/train_images.npy", train_images)

print("saved:", "onnx/train_images.npy")
print("shape:", train_images.shape)
print("dtype:", train_images.dtype)
print("min/max:", train_images.min(), train_images.max())

In [ ]:
import torch 
from torch import nn

class DummyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(16, 10)

    def forward(self, x):
        x = self.conv(x)
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x
    
model = DummyModel()
dummy_input = torch.randn(1, 3, 640, 640)

torch.onnx.export(
    model,
    dummy_input,
    "onnx/dummy_model.onnx",
    opset_version=9,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["output"],
    dynamo=None,
    dynamic_axes=None,
)

In [ ]:
import yaml

from ultralytics.nn.tasks import DetectionModel

default_cfg = yaml.load(open("ultralytics/cfg/default.yaml", "r"), Loader=yaml.FullLoader)
model_cfg = yaml.load(open("cfgs/yolov12.yaml", "r"), Loader=yaml.FullLoader)
model_cfg["scale"] = "n"

model = DetectionModel(cfg=model_cfg, ch=3, verbose=False)
print(model)